# XGBoost: Choose Between `UMKP-P` and `UMKP-CTCP-P`

This notebook reads the `all` sheet from `results-filtered.xlsx`, engineers the structural features,
and trains an XGBoost classifier to predict which algorithm is faster:

- `UMKP-P`
- `UMKP-CTCP-P`

The target is built from the BnB columns:

- `UMKP-P-bnb`
- `UMKP-CTCP-P-bnb`


In [59]:
from ast import literal_eval
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

NOTEBOOK_DIR = Path.cwd()
FILE_PATH = NOTEBOOK_DIR / "results-filtered.xlsx"
SHEET_NAME = "all"
TARGET_A = "UMKP-P-bnb"
TARGET_B = "UMKP-CTCP-P-bnb"
FEATURE_BASE_COLS = ["dataset", "k", "n", "m", "d_avg", "d_max", "core_number", "shell_sizes"]

In [60]:
def parse_shell_sizes(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    return literal_eval(text)


def shell_features(shell_value):
    shells = parse_shell_sizes(shell_value)
    if not shells:
        return {
            "core_min": np.nan,
            "shell_num": 0,
            "shell_avgSize": np.nan,
            "shell_maxSize": np.nan,
            "shell_minSize": np.nan,
            "shell_stdSize": np.nan,
        }

    sizes = [size for _, size in shells]
    return {
        "core_min": shells[0][0],
        "shell_num": len(shells),
        "shell_avgSize": float(np.mean(sizes)),
        "shell_maxSize": int(np.max(sizes)),
        "shell_minSize": int(np.min(sizes)),
        "shell_stdSize": float(np.std(sizes)),
    }


def build_feature_table(df_raw):
    df_base = df_raw[FEATURE_BASE_COLS + [TARGET_A, TARGET_B]].copy()
    shell_df = pd.DataFrame(df_base["shell_sizes"].apply(shell_features).tolist())

    df_features = pd.concat([
        df_base[["dataset", "k", "n", "m", "d_avg", "d_max"]].copy(),
        df_base[["core_number"]].rename(columns={"core_number": "core_max"}),
        shell_df,
        df_base[[TARGET_A, TARGET_B]].copy(),
    ], axis=1)
    return df_features


In [61]:
df_raw = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
df = build_feature_table(df_raw)

df.head()

,dataset,k,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize,UMKP-P-bnb,UMKP-CTCP-P-bnb
0,12month1,20,52328,1585220,30.294,9214,57,19,39,16785.102564,52328,1030,14354.722410,56.8765,38.7548
1,130bit,8,584,6059,10.375,322,12,8,5,564.400000,584,509,28.443628,0.7182,9.5405
2,130bit,9,584,6059,10.375,322,12,8,5,564.400000,584,509,28.443628,2.562,53.8994
3,130bit,10,583,6051,10.379,321,12,9,4,559.500000,583,509,29.853810,2.1395,46.5601
4,144,4,144319,1072349,7.430,26,9,7,3,140266.333333,144319,132943,5188.210310,0.4818,0.4362


In [62]:
x_mask_a = df[TARGET_A].astype(str).str.strip().str.lower().eq("x")
x_mask_b = df[TARGET_B].astype(str).str.strip().str.lower().eq("x")

df.loc[x_mask_a, TARGET_A] = 1800
df.loc[x_mask_b, TARGET_B] = 1800

df[TARGET_A] = pd.to_numeric(df[TARGET_A], errors="coerce").fillna(1800)
df[TARGET_B] = pd.to_numeric(df[TARGET_B], errors="coerce").fillna(1800)

tie_mask = df[TARGET_A] == df[TARGET_B]
df_model = df.loc[~tie_mask].copy()

df_model["target_name"] = np.where(
    df_model[TARGET_A] < df_model[TARGET_B],
    "UMKP-P",
    "UMKP-CTCP-P",
)
df_model["target"] = np.where(df_model["target_name"] == "UMKP-P", 0, 1)

print("All rows:", len(df))
print("Replaced x rows in TARGET_A:", int(x_mask_a.sum()))
print("Replaced x rows in TARGET_B:", int(x_mask_b.sum()))
print("Dropped exact ties:", int(tie_mask.sum()))
print("Training rows:", len(df_model))
print()
print(df_model["target_name"].value_counts())

All rows: 4356
Replaced x rows in TARGET_A: 324
Replaced x rows in TARGET_B: 530
Dropped exact ties: 593
Training rows: 3763

target_name
UMKP-P         2073
UMKP-CTCP-P    1690
Name: count, dtype: int64


In [63]:
feature_cols = [
    "k",
    "n",
    "m",
    "d_avg",
    "d_max",
    "core_max",
    "core_min",
    "shell_num",
    "shell_avgSize",
    "shell_maxSize",
    "shell_minSize",
    "shell_stdSize",
]

X = df_model[feature_cols].copy()
y = df_model["target"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (3010, 12)
Test shape: (753, 12)


In [64]:
model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [65]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

label_map = {0: "UMKP-P", 1: "UMKP-CTCP-P"}

print(f"Accuracy: {acc:.4f}")
print()
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=[label_map[0], label_map[1]]))

Accuracy: 0.8645

Confusion matrix:
[[369  46]
 [ 56 282]]

Classification report:
              precision    recall  f1-score   support

      UMKP-P       0.87      0.89      0.88       415
 UMKP-CTCP-P       0.86      0.83      0.85       338

    accuracy                           0.86       753
   macro avg       0.86      0.86      0.86       753
weighted avg       0.86      0.86      0.86       753



In [66]:
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

importance_df

,feature,importance
2,m,0.235122
8,shell_avgSize,0.098405
5,core_max,0.082459
4,d_max,0.074457
1,n,0.071991
6,core_min,0.066496
10,shell_minSize,0.065603
3,d_avg,0.064204
7,shell_num,0.063869
9,shell_maxSize,0.062932


In [67]:
test_results = X_test.copy()
test_results["actual"] = y_test.map(label_map)
test_results["predicted"] = pd.Series(y_pred, index=y_test.index).map(label_map)
test_results["correct"] = test_results["actual"] == test_results["predicted"]

test_results = test_results.join(df_model[["dataset", TARGET_A, TARGET_B]], how="left")
test_results = test_results[["dataset", "actual", "predicted", "correct", TARGET_A, TARGET_B] + feature_cols]

test_results.head(20)

,dataset,actual,predicted,correct,UMKP-P-bnb,UMKP-CTCP-P-bnb,k,n,m,d_avg,d_max,core_max,core_min,shell_num,shell_avgSize,shell_maxSize,shell_minSize,shell_stdSize
2138,misc-Reuters911,UMKP-P,UMKP-P,True,0.1710,0.3486,9,154,7061,45.851,153,66,48,12,143.083333,154,127,9.232717
1135,edit-svwiki,UMKP-P,UMKP-P,True,30.0789,1800.0000,5,829,66360,80.048,792,104,44,61,663.918033,829,379,128.435006
1985,lkml-reply,UMKP-P,UMKP-P,True,1.2169,3.2790,20,186,11701,62.909,185,91,75,14,173.428571,186,157,9.424241
2157,mono_500Hz,UMKP-P,UMKP-P,True,4.9764,35.9737,10,719,26733,37.181,718,39,39,1,719.000000,719,719,0.000000
4089,venkat50,UMKP-P,UMKP-P,True,0.5870,0.6014,4,62424,827684,13.259,43,19,15,2,62278.000000,62424,62132,146.000000
1642,gyro_k,UMKP-CTCP-P,UMKP-CTCP-P,True,0.5198,0.3806,15,13584,430230,31.672,359,41,32,8,11724.250000,13584,7989,2165.676670
799,crankseg_2,UMKP-CTCP-P,UMKP-CTCP-P,True,2.5395,1.6931,7,20838,1554042,74.577,290,101,86,6,13177.500000,20838,2712,6989.513663
3135,sme3Da,UMKP-CTCP-P,UMKP-P,False,0.8731,0.6904,20,12432,429385,34.539,344,44,29,8,10621.875000,12432,3852,2624.988402
890,degme,UMKP-CTCP-P,UMKP-CTCP-P,True,749.1859,475.8824,10,213559,2051668,9.607,157988,13,9,5,170953.200000,213559,30901,70595.883065
39,3Dspectralwave2,UMKP-CTCP-P,UMKP-CTCP-P,True,0.9528,0.7550,10,23685,486513,20.541,92,27,20,6,19488.666667,23685,18120,1927.094935


In [68]:
from openpyxl import load_workbook
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

def format_excel_table(path, sheet_name="Sheet1", table_name="Table1"):
    wb = load_workbook(path)
    ws = wb[sheet_name]
    max_row = ws.max_row
    max_col = ws.max_column
    if max_row >= 1 and max_col >= 1:
        end_col = get_column_letter(max_col)
        table = Table(displayName=table_name, ref=f"A1:{end_col}{max_row}")
        table.tableStyleInfo = TableStyleInfo(
            name="TableStyleMedium9",
            showFirstColumn=False,
            showLastColumn=False,
            showRowStripes=True,
            showColumnStripes=False,
        )
        ws.add_table(table)
        header_font = Font(size=12, bold=True)
        body_font = Font(size=11)
        for cell in ws[1]:
            cell.font = header_font
        for row in ws.iter_rows(min_row=2, max_row=max_row, max_col=max_col):
            for cell in row:
                cell.font = body_font
        for col_idx in range(1, max_col + 1):
            letter = get_column_letter(col_idx)
            max_len = max(len(str(ws.cell(row=r, column=col_idx).value or "")) for r in range(1, max_row + 1))
            ws.column_dimensions[letter].width = min(max(max_len + 2, 10), 40)
    wb.save(path)

OUT_XLSX = NOTEBOOK_DIR / "xgboost_umkp_p_vs_ctcp_p_test_predictions.xlsx"
OUT_IMPORTANCE = NOTEBOOK_DIR / "xgboost_umkp_p_vs_ctcp_p_feature_importance.xlsx"

test_results_export = test_results.reset_index().rename(columns={"index": "row_id"})
test_results_export.to_excel(OUT_XLSX, index=False)
importance_df.to_excel(OUT_IMPORTANCE, index=False)
format_excel_table(OUT_XLSX, table_name="PredictionsTable")
format_excel_table(OUT_IMPORTANCE, table_name="ImportanceTable")

print("Wrote:")
print(OUT_XLSX)
print(OUT_IMPORTANCE)

Wrote:
/Users/akahmad/Documents/kplex-ML/notebooks/xgboost_umkp_p_vs_ctcp_p_test_predictions.xlsx
/Users/akahmad/Documents/kplex-ML/notebooks/xgboost_umkp_p_vs_ctcp_p_feature_importance.xlsx


In [69]:
tolerances = [0, 1, 2, 5, 10, 30]

def predicted_runtime(row):
    return row[TARGET_A] if row["predicted"] == "UMKP-P" else row[TARGET_B]

def fastest_runtime(row):
    return min(row[TARGET_A], row[TARGET_B])

pred_times = test_results.apply(predicted_runtime, axis=1)
fast_times = test_results.apply(fastest_runtime, axis=1)

tolerance_rows = []
for tol in tolerances:
    ok = pred_times <= fast_times + tol
    tolerance_rows.append({
        "tolerance_sec": tol,
        "correct_count": int(ok.sum()),
        "total_rows": len(test_results),
        "accuracy_percent": float(ok.mean() * 100),
    })

tolerance_df = pd.DataFrame(tolerance_rows)
tolerance_df["accuracy_percent"] = tolerance_df["accuracy_percent"].round(2)
tolerance_df

,tolerance_sec,correct_count,total_rows,accuracy_percent
0,0,651,753,86.45
1,1,722,753,95.88
2,2,725,753,96.28
3,5,732,753,97.21
4,10,734,753,97.48
5,30,736,753,97.74
